In [ ]:
import EL
from typing import Dict, Any, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
sns.set()

# Simulation Exercise 1: Inference on the Mean


- DGP: $$X \sim (1-\epsilon) N(\mu, 1)+ \epsilon N(\mu+\delta, \sigma_c^2), \quad \theta=\mu$$

- Moment Conditions:
$$g(x,\theta) = x-\theta$$

- In simulation, I let $\epsilon = 0.05$, $\delta=8$, $\sigma_c = 3$, $\mu = 0$. $n=500$ samples.

In [ ]:
# Simulate DGP, and run generate EL

def simulate_contaminated_mean(
    n: int,
    mu: float = 0.0,
    eps: float = 0.05,
    delta: float = 8.0,
    sig_c: float = 3.0,
    seed: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns:
      X: (N,1)
      theta0: (1,)  robust init (median)
      theta_true: (1,)  the DGP mean mu
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(loc=mu, scale=1.0, size=n)
    k = int(round(eps * n))
    if k > 0:
        idx = rng.choice(n, size=k, replace=False)
        X[idx] = rng.normal(loc=mu + delta, scale=sig_c, size=k)
    X = X.reshape(-1, 1)
    theta_true = (1-eps) * mu + eps * (mu+delta)
    return X, theta_true

def g_mean(X: np.ndarray, theta: np.ndarray) -> np.ndarray:
    return X - theta  # (N,1)


X1, theta_true = simulate_contaminated_mean(n=500, mu=0, eps=0.10, delta=8, sig_c=3.0, seed=0)
EL1_mean = EL.GenELV2(
    X = X1,
    g = g_mean,
    theta0 = np.array([np.mean(X1)]),
    init_theta = "etel",
    alpha = 0.0,
    m = 0,
    B_nums = 500,
    random_state= 42,
    bounds = None,
    newton_tol = 1e-10,
    newton_maxiter = 100,
    newton_ridge = 1e-10,
    per_v_gmm_start= True
)
result1 = EL1_mean.fit()  # internally samples v via bayesian_bootstrap()

In [ ]:
ci = (2.5, 97.5)
lo, hi = np.nanpercentile(result1.thetas, ci)
pos_mean = np.nanmean(result1.thetas)
plt.figure(figsize=(7,5))
plt.hist(result1.thetas, alpha=0.6)
plt.axvline(lo, linestyle="--", linewidth=2, color='red', label=f"GEL {ci[0]}% = {lo:.3g}")
plt.axvline(hi, linestyle="--", linewidth=2, color='red',label=f"GEL {ci[1]}% = {hi:.3g}")
plt.axvline(pos_mean, linestyle="-.", linewidth=2, label=f"GEL mean = {pos_mean:.3g}")
plt.axvline(theta_true, linewidth=2, label=f"true = {theta_true:.3g}")
plt.title("Generative EL Postrior Distribution on The mean (500 Boostrap Draws)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

# Simulation Exercise 2: overidentified linear IV with endogeneity.

- DGP: Consider an over-identification scenario: let $z_1$, $z_2$ be instrumental variables, $u$ (structural) and $v$ (first-stage) be errors such that $(z_1, z_2) \bot (u,v)$. If $cov(u,v) \neq 0$, then $cov(x,u) \neq 0$, then OLS is biased. 
\begin{align}
x &= \pi_1 z_1 + \pi_2 z_2 + v \\
y &= \beta x +u 
\end{align}
- We would like to target $\beta$ so the moment conditions are
$$E[z_1(y-\beta x)]=0, \quad E[z_2(y-\beta x)]=0$$

- Parameter choice: $\beta=1$, $\pi_1=0.8, \pi_2=0.6, \rho_{uv} = 0.4$.

In [ ]:
# Simulation DGP

def simulate_iv_overidentified(
    n: int,
    beta: float = 1.0,
    pi1: float = 0.8,
    pi2: float = 0.6,
    rho_uv: float = 0.4,
    seed: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    DGP:
      x = pi1*z1 + pi2*z2 + v
      y = beta*x + u
    (z1,z2) ~ N(0,1) i.i.d., (u,v) jointly normal with corr rho_uv.
    Returns:
      X: (N,4) with columns [y, x, z1, z2]
      theta0: (1,) initial beta (OLS as a cheap start)
      theta_true: (1,) true beta used in simulation
    """
    rng = np.random.default_rng(seed)

    # Instruments
    z1 = rng.normal(size=n)
    z2 = rng.normal(size=n)

    # Correlated (u, v): build via correlation
    e1 = rng.normal(size=n)
    e2 = rng.normal(size=n)
    u = e1
    v = rho_uv * e1 + np.sqrt(max(0.0, 1.0 - rho_uv**2)) * e2

    # Endogenous regressor and outcome
    x = pi1 * z1 + pi2 * z2 + v
    y = beta * x + u

    X = np.column_stack([y, x, z1, z2]).astype(float)

    # Simple initial guess for beta: OLS slope (biased under endogeneity, but good numerically)
    denom = float(np.dot(x, x))
    beta_ols = float(np.dot(x, y) / denom) if denom > 1e-12 else 0.0
    theta_OLS = np.array([beta_ols], dtype=float)

    theta_true = np.array([beta], dtype=float)
    return X, theta_OLS, theta_true


def g_iv_two_instruments(X: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """
    IV moment function for two instruments:
      g_i(beta) = [ z1_i * (y_i - beta * x_i),  z2_i * (y_i - beta * x_i) ]
    Returns (N, 2).
    """
    y, x, z1, z2 = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
    resid = y - theta[0] * x
    return np.column_stack([z1 * resid, z2 * resid])



X2, theta_ols, theta_true = simulate_iv_overidentified(n=500, beta=1.0, pi1=0.8, pi2=0.6, rho_uv=0.4, seed=0)


EL2_IV = EL.GenELV2(
    X = X2,
    g =  g_iv_two_instruments,
    theta0 = np.array([theta_ols[0]]),
    alpha = 0.0,
    m = 0,
    B_nums = 500,
    random_state= 42,
    bounds = None,
    newton_tol = 1e-10,
    newton_maxiter = 100,
    newton_ridge = 1e-10,
    per_v_gmm_start= True
)
result2 = EL2_IV.fit()  

In [ ]:
ci = (2.5, 97.5)
lo, hi = np.nanpercentile(result2.thetas, ci)
pos_mean = np.nanmean(result2.thetas)

plt.figure(figsize=(7,5))
plt.hist(result2.thetas, alpha=0.6)
plt.axvline(lo, linestyle="--", linewidth=2, color='red', label=f"CI {ci[0]}% = {lo:.3g}")
plt.axvline(hi, linestyle="--", linewidth=2, color='red', label=f"CI {ci[1]}% = {hi:.3g}")
plt.axvline(pos_mean, linestyle="-.", linewidth=2, label=f"Posterior mean = {pos_mean:.3g}")
plt.axvline(theta_true[0], linewidth=2, label=f"true = {theta_true[0]:.3g}")
plt.axvline(theta_ols[0], linewidth=2, color="yellow", label=f"OLS beta = {theta_ols[0]:.3g}")
plt.title("BETEL Bootstrap Posterior Distribution on IV Regression Coefficient (500 Bootstrap Draws)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()

plt.savefig("betel_posterior_iv.pdf", bbox_inches="tight")
plt.show()

# Simulation Exercise 3: Logistic Regression, "higher" dimension

- DGP: Logistic Regression, binary observation $y_i$
- Moment Conditions: $$E[\beta(y_i-\sigma(\beta' x_i))], \quad \sigma(x) = \frac{1}{1+e^{-x}}$$
- Choice of paramters: $\beta_0 = [-0.3, 1.0, -0.8]$ (including intercept), $\rho_{x_1,x_2} = 0.5$, $n=500$

In [ ]:
def simulate_logistic_vecparam(
    n: int,
    beta_true: np.ndarray = np.array([-0.3, 1.0, -0.8]),  # [intercept, b1, b2]
    rho_x: float = 0.5,                                   # corr(X1, X2)
    seed: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    DGP:
      [X1, X2] ~ mean 0, corr = rho_x
      P(Y=1 | X) = sigmoid( [1, X1, X2] @ beta_true )
    Returns:
      X: (N, 1 + p) + 1 = (N, 4) with columns [y, 1, X1, X2]
      theta0: (p,) starting guess
      theta_true: (p,) true parameter
    """
    rng = np.random.default_rng(seed)

    # Build correlated features (X1, X2) with Corr = rho_x
    e1 = rng.normal(size=n)
    e2 = rng.normal(size=n)
    X1 = e1
    X2 = rho_x * e1 + np.sqrt(max(0.0, 1.0 - rho_x**2)) * e2

    F = np.column_stack([np.ones(n), X1, X2])  # (N, p=3)
    logits = F @ beta_true
    p = 1.0 / (1.0 + np.exp(-logits))
    y = rng.binomial(1, p, size=n)

    X = np.column_stack([y, F]).astype(float)  # [y, 1, X1, X2]
    theta_true = beta_true.astype(float)
    return X,  theta_true

def g_logit_score(X: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """
    Logistic score moments:
      g_i(theta) = F_i * (y_i - sigmoid(F_i^T theta))  -> shape (N, p)
    X columns: [y, 1, X1, X2]; theta shape: (p,)
    """
    y = X[:, 0]
    F = X[:, 1:]                    # (N, p)
    logits = F @ theta
    p = 1.0 / (1.0 + np.exp(-logits))
    return F * (y - p)[:, None]     # (N, p)


X3, theta_true = simulate_logistic_vecparam(n=500, seed=0)

#Fit 

EL3_logit = EL.GenELV2(
    X = X3,
    g =  g_logit_score,
    theta0 = np.zeros(3),
    alpha = 0.0,
    m = 0,
    B_nums = 500,
    random_state= 42,
    bounds = None,
    newton_tol = 1e-10,
    newton_maxiter = 100,
    newton_ridge = 1e-10,
    per_v_gmm_start= True
)
result3 = EL3_logit.fit()  # internally 


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
ci = (2.5, 97.5)

lo, hi = np.nanpercentile(result3.thetas, ci, axis=0)
pos_mean = np.nanmean(result3.thetas, axis=0) 

for i in range(3):
    axes[i].hist(result3.thetas[:, i], alpha=0.6)

    axes[i].axvline(lo[i], linestyle="--", linewidth=2, color="red",
                    label=f"GEL {ci[0]}% = {lo[i]:.3g}")
    axes[i].axvline(hi[i], linestyle="--", linewidth=2, color="red",
                    label=f"GEL {ci[1]}% = {hi[i]:.3g}")

    axes[i].axvline(pos_mean[i], linestyle="-.", linewidth=2,
                    label=f"GEL mean = {pos_mean[i]:.3g}")

    axes[i].axvline(theta_true[i], linewidth=2,
                    label=f"true = {theta_true[i]:.3g}")

    axes[i].set_title(f"Coefficient {i} (500 Bootstrap Draws)")
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Density")
    axes[i].legend()

fig.tight_layout()
plt.show()


# Redo Experiment 2 (Larger Sample, Paper Version)

In [ ]:
def summarize_draws(
    draws: np.ndarray,
    lower_q: float = 0.05,
    upper_q: float = 0.95,
) -> dict:
    draws = np.asarray(draws, dtype=float).reshape(-1)
    draws = draws[np.isfinite(draws)]

    return {
        "Mean": np.mean(draws),
        "SD": np.std(draws, ddof=1),
        "Median": np.median(draws),
        "Lower": np.quantile(draws, lower_q),
        "Upper": np.quantile(draws, upper_q),
    }
# ---------------------------------------------------------------------
# Run one simulated dataset
# ---------------------------------------------------------------------

n = 500
beta_true = 1.0
pi1 = 0.8
pi2 = 0.6
rho_uv = 0.4
B = 5000

X_iv, beta_ols, beta_ols_plim = simulate_iv_overidentified(
    n=n,
    beta=beta_true,
    pi1=pi1,
    pi2=pi2,
    rho_uv=rho_uv,
    seed=0,
)
print(beta_ols, beta_ols_plim)

EL_iv = EL.GenELV2(
    X=X_iv,
    g=g_iv_two_instruments,
    theta0=np.array([beta_ols[0]]),
    alpha=0.0,
    m=0,
    B_nums=B,
    random_state=42,
    bounds=None,
    newton_tol=1e-8,
    newton_maxiter=100,
    newton_ridge=1e-10,
    per_v_gmm_start=True,
)

result_iv = EL_iv.fit()

posterior_draws = np.asarray(result_iv.thetas).reshape(-1)
summary = summarize_draws(posterior_draws, lower_q=0.05, upper_q=0.95)

print("True beta:", beta_true)
print("Sample OLS:", beta_ols)
print("Population OLS plim:", beta_ols_plim)
print(summary)



# ---------------------------------------------------------------------
# Plot posterior draws
# ---------------------------------------------------------------------
lower = summary["Lower"]
upper = summary["Upper"]
post_mean = summary["Mean"]

plt.figure(figsize=(7, 4.8))

plt.hist(
    posterior_draws,
    bins=20,
    density=True,
    alpha=0.65,
    edgecolor="white",
)

plt.axvline(lower, linestyle="--", linewidth=2, label="5% posterior quantile")
plt.axvline(upper, linestyle="--", linewidth=2, label="95% posterior quantile")
plt.axvline(post_mean, linestyle="-.", linewidth=2, label="Posterior mean")
plt.axvline(beta_true, linewidth=2, label="True value")
plt.axvline(beta_ols, linewidth=2, label="Sample OLS")

plt.xlabel(r"$\beta$")
plt.ylabel("Density")
plt.title("Posterior draws in the overidentified IV simulation")
plt.legend(frameon=True)
plt.tight_layout()

plt.savefig("images/etel_posterior_iv.pdf", bbox_inches="tight")
plt.show()